<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/DL-2026/%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_%D0%B4%D0%BB%D1%8F_%D1%81%D1%82%D1%83%D0%B4%D0%B5%D0%BD%D1%82%D0%BE%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 **Задание для студентов: Разработка интеллектуальной системы с мультиагентной архитектурой и RAG**

---

## 🎯 **Цель проекта**

Создать **интеллектуальную систему** на любую тему (на выбор студента): машинный перевод, генерация сказок, QA-система, анализ резюме, рекомендации, суммаризация, чат-бот, проверка кода и т.д. Главное – применить современный стек технологий и архитектурные паттерны.

**Система должна включать:**
- Мультиагентную архитектуру (минимум 3 агента + Supervisor)
- Retrieval-Augmented Generation (RAG) с гибридным поиском (BM25 + векторный поиск)
- Два варианта LLM: Ollama и HuggingFace Transformers
- REST API (FastAPI)
- Асинхронную обработку, кэширование, фоновые задачи (Celery)
- Human-in-the-Loop (обратная связь и обработка неоднозначных результатов)
- Контейнеризацию (Docker)
- Тестирование (pytest)
- Простой UI (Streamlit или аналог)

**Важно:** студенты **не** используют готовые фреймворки для агентов (LangChain, LangGraph и т.п.). Вся логика агентов и оркестрации реализуется самостоятельно. Готовый код не предоставляется, даётся только описание архитектуры и требований.

---

## 🏗️ **Примерная архитектура проекта (ориентир)**

Ниже приведена структура, которую можно взять за основу. Студенты вольны менять названия файлов и папок, но обязаны реализовать все перечисленные компоненты.

```
project/
├── backend/
│   ├── agents/            # Агенты и супервизор
│   │   ├── base.py        # Абстрактный класс Agent
│   │   ├── validator.py   # Агент проверки (rule-based + LLM)
│   │   ├── critic.py      # Агент оценки качества (LLM-as-Judge)
│   │   ├── editor.py      # Агент исправления/улучшения
│   │   └── supervisor.py  # Оркестратор цикла
│   ├── api/
│   │   ├── routes/        # Роуты FastAPI
│   │   ├── deps.py        # Зависимости (auth, DB)
│   │   └── main.py        # Точка входа приложения
│   ├── core/
│   │   ├── config.py      # Настройки (pydantic-settings)
│   │   ├── constants.py   # Константы
│   │   ├── models.py      # Pydantic-модели
│   │   └── exceptions.py  # Свои исключения
│   ├── data/
│   │   ├── knowledge_base.py    # Работа с базой знаний / справочниками
│   │   ├── examples_store.py    # Работа с коллекцией примеров
│   │   ├── cache.py       # Кэш
│   │   ├── embeddings.py  # Эмбеддинги (SentenceTransformer)
│   │   ├── llm_manager.py # Выбор между Ollama и HuggingFace
│   │   └── hf_model.py    # Обёртка для HF pipeline
│   ├── db/
│   │   ├── models.py      # SQLAlchemy-модели таблиц
│   │   ├── session.py     # Подключение к БД
│   │   └── database.py    # Экспорт
│   ├── hitl/              # Human-in-the-Loop
│   │   ├── active_learning.py # Очередь неоднозначных примеров
│   │   ├── feedback.py    # Обратная связь
│   │   └── updater.py     # Обновление базы знаний
│   ├── retrieval/
│   │   ├── bm25_index.py  # BM25
│   │   ├── vector_index.py# Векторный индекс (Qdrant)
│   │   ├── hybrid_retriever.py # Гибридный поиск + RRF
│   │   └── reranker.py    # (опционально) переранжирование
│   ├── utils/
│   │   ├── llm_client.py  # Обёртка для LLM
│   │   ├── llm_processing.py # Очистка ответов
│   │   ├── async_utils.py # Retry-декоратор
│   │   ├── logging.py     # Настройка логирования
│   │   ├── security.py    # JWT, пароли
│   │   └── text_processing.py # Токенизация, нормализация
│   └── worker/            # Celery
│       ├── celery_app.py  # Приложение Celery
│       └── tasks.py       # Задачи
├── frontend/              # ИЛИ streamlit_app/
│   ├── app.py             # Главная страница
│   ├── pages/             # Дополнительные страницы
│   └── utils/
│       ├── api_client.py  # Клиент для API
│       └── constants.py   # Справочники
├── scripts/
│   ├── create_admin.py    # Создание админа
│   ├── import_data.py     # Массовый импорт CSV/JSON
│   ├── run_eval.py        # Скрипт оценки качества
│   └── import/            # Скрипты добавления данных
├── tests/
│   ├── unit/              # Модульные тесты
│   └── integration/       # Интеграционные тесты
├── infrastructure/
│   ├── docker-compose.yml # Все сервисы
│   ├── Dockerfile         # Образ приложения
│   └── nginx.conf         # Прокси
├── data/
│   └── import/            # CSV/JSON-файлы с эталонными данными
├── pyproject.toml         # Poetry
└── run.py                 # Точка входа
```

---

## 🤖 **Требования к мультиагентной системе**

1. **Базовый класс агента**
   - Абстрактный класс `Agent` с методом `async def process(state) -> state`.
   - Хранит имя агента и последний результат.
   - Все агенты наследуются от него.

2. **Валидатор (Validator)**
   - Выполняет проверку результата по формальным правилам (минимальная длина, наличие обязательных полей, отсутствие повторов и т.п.).
   - Может вызывать LLM для дополнительной проверки.
   - Возвращает список ошибок (`errors`) и предупреждений (`warnings`), а также флаг `passed`.

3. **Критик (Critic)**
   - LLM-as-a-Judge: оценивает результат по нескольким критериям (например, релевантность, полнота, стиль, точность).
   - Использует few-shot prompting с примерами хороших и плохих ответов.
   - Возвращает баллы (0–10) по каждому критерию, общий балл и рекомендации.
   - В случае ошибки LLM (таймаут, неверный JSON) возвращает fallback-оценку (например, 5.0).

4. **Редактор (Editor)**
   - Принимает ошибки/предупреждения от Validator и рекомендации от Critic.
   - Исправляет или улучшает результат через LLM.
   - Возвращает исправленный результат и флаг, изменилось ли что-то.

5. **Супервизор (Supervisor)**
   - Оркеструет работу агентов в цикле:
     ```
     Validator -> Critic -> (если нужно) Editor -> Validator -> Critic -> ...
     ```
   - Ограничивает количество итераций (`max_iterations`).
   - Определяет критерий успеха (например, `validation_passed == True` и `critic_score >= 7`).
   - Ведёт лог всех шагов (`reasoning_log`).

---

## 🔍 **Требования к RAG**

1. **Гибридный поиск**
   - Реализовать два индекса: BM25 (лексический) и векторный (семантический, Qdrant).
   - Объединить результаты методом Reciprocal Rank Fusion (RRF).

2. **Эмбеддинги**
   - Использовать `sentence-transformers` (например, `paraphrase-multilingual-MiniLM-L12-v2`).
   - Оборачивать в класс-синглтон с ленивой загрузкой.

3. **Векторная БД**
   - Qdrant (или аналог: ChromaDB, FAISS) с коллекцией и cosine-метрикой.
   - Добавление, поиск с фильтрами по категориям/языкам (если применимо).

4. **Интеграция с LLM**
   - Контекст из найденных записей подаётся в промпт генеративной модели.

---

## 🧠 **Поддержка LLM: Ollama и HuggingFace**

1. **Ollama** — основной вариант для локального запуска.
   - HTTP API (`/api/generate`).
   - Поддержка разных моделей (например, qwen2.5, mistral, llama3.1).
   - Настройка `temperature` для разных агентов.

2. **HuggingFace Transformers** — запасной вариант для случаев, когда нужна конкретная модель или когда Ollama недоступен.
   - Асинхронная обёртка над `transformers.pipeline`.
   - Кэширование результатов.

3. **Выбор бэкенда** (через `LLMManager` или аналогичный класс)
   - Если задача поддерживается Ollama — использовать его.
   - Иначе переключаться на HuggingFace.

---

## 📊 **Сбор эталонных данных (500–1000+ записей)**

Студенты должны подготовить **обучающий набор** для базы знаний и справочников (аналог глоссария).

- **Справочник / база терминов**: не менее **500** уникальных записей (пары или структурированные объекты).
- **База знаний / коллекция примеров**: не менее **1000** записей (пары "вход-выход", "вопрос-ответ", "текст-резюме" и т.п.).
- Данные можно взять из открытых источников (например, датасеты с Kaggle, OPUS, Tatoeba, готовые корпуса) или сгенерировать/составить вручную.
- Формат CSV или JSON. Колонки зависят от предметной области, но должны быть согласованы с кодом импорта.

**Скрипт генерации**:
- Читает исходные файлы (JSON, CSV или txt).
- Сэмплирует нужное количество записей.
- При необходимости подсчитывает частотность и отбирает самые частотные для справочника.
- Сохраняет результат в CSV/JSON.

**Скрипт импорта** в БД:
- Читает CSV/JSON.
- Вставляет записи в таблицы через SQLAlchemy, избегая дубликатов.

---

## 🛠️ **Backend требования**

1. **FastAPI** со следующими группами эндпоинтов (названия адаптируются под задачу):
   - Основной запрос (например, `/generate`, `/answer`, `/analyze`).
   - Мульти-запрос (несколько вариантов/языков/категорий).
   - Пакетная обработка.
   - Ансамбль моделей с выбором лучшего варианта.
   - Административные эндпоинты для управления справочником и базой знаний.
   - Аутентификация: регистрация, вход, получение информации.
   - Оценка качества (BLEU, chrF, BERTScore, METEOR для текстовых задач или свои метрики).
   - Human-in-the-Loop: получение неоднозначных примеров, подтверждение/отклонение.
   - Проверка состояния сервисов (`/health`).

2. **Аутентификация**:
   - JWT токены.
   - Роли: `user`, `editor`, `admin`.
   - Хэширование паролей (bcrypt).

3. **База данных**:
   - SQLAlchemy (async) с поддержкой PostgreSQL (основной) и SQLite (для тестов).
   - Таблицы: пользователи, справочник, база знаний, неоднозначные примеры, кэш.

4. **Кэширование**:
   - Таблица в БД или Redis.
   - Ключ: например, `(input_text, category, parameters)`.
   - Увеличивать счётчик попаданий.

5. **Фоновые задачи (Celery)**:
   - Переиндексация базы знаний.
   - Пакетная обработка.
   - Очистка устаревшего кэша.
   - Массовое добавление записей.

6. **Логирование**:
   - Структурированные логи (JSON optional).
   - Ротация файлов.
   - Разные уровни для разных модулей.

---

## 🧪 **Тестирование**

- Использовать **pytest** и **pytest-asyncio**.
- Покрытие тестами не менее **70%** (можно проверить `pytest-cov`).
- Обязательные категории:
  - Юнит-тесты для каждого агента (проверка логики, обработка ошибок).
  - Юнит-тесты для retrieval (BM25, vector, RRF).
  - Юнит-тесты для API-роутов (с mock-объектами).
  - Интеграционные тесты (через `TestClient`).
- Использовать `monkeypatch` и `unittest.mock` для изоляции внешних сервисов (Ollama, Qdrant, Redis).
- Создавать фикстуры для тестовой БД и тестовых данных.

---

## 🐳 **Инфраструктура**

1. **Poetry** для управления зависимостями (`pyproject.toml`).
2. **Docker Compose** со следующими сервисами:
   - `db` — PostgreSQL 16
   - `redis` — Redis 7
   - `qdrant` — векторная БД
   - `ollama` — LLM (с инициализацией и подгрузкой моделей)
   - `app` — FastAPI
   - `worker` — Celery worker
   - `beat` — Celery beat
   - `nginx` — прокси
3. **Healthcheck** для каждого сервиса.
4. **Nginx** конфигурация с проксированием, gzip, security headers.
5. **Dockerfile** для сборки образа приложения.

---

## 🖥️ **Frontend (Streamlit или аналог)**

- Страница входа/регистрации.
- Основная страница с формой для выполнения задачи (генерация/ответ/анализ).
- Страница пакетной обработки (загрузка файла, массовая обработка).
- Страница управления справочником и базой знаний (CRUD, импорт CSV).
- Страница Human-in-the-Loop (просмотр неоднозначных примеров, подтверждение/отклонение).
- Страница оценки качества (запуск метрик).
- Страница мониторинга (статусы сервисов, статистика кэша).

---

## 📋 **Критерии оценки**

| Критерий | Вес |
|----------|-----|
| Архитектура и чистота кода | 30% |
| Функциональность (работающие агенты, RAG, API) | 30% |
| Тестирование (покрытие, качество) | 20% |
| Документация (README, инструкции) | 10% |
| Инфраструктура (Docker, Poetry) | 10% |

**Бонусы** (до +10%): мониторинг, CI/CD, WebSocket, rate limiting, A/B тесты.

---

## 📦 **Итоговый результат**

Студент должен предоставить:

1. Репозиторий с полным кодом.
2. README с инструкциями по запуску (через Docker одной командой).
3. Набор эталонных данных (CSV/JSON) объёмом от 500 до 1000+ записей.
4. Скрипты для генерации и импорта данных.
5. Скрипт `run_eval.py` для оценки качества и сравнения режимов (baseline, +справочник, +RAG, +agents).
6. Отчёт с описанием архитектуры и принятых решений.

---

**Удачи!** Помните: главное — не просто скопировать идею, а понять принципы и реализовать свою систему.

# 🗓️ **Поэтапный план разработки (15 недель)**

Каждый этап рассчитан на одну неделю. В конце этапа вы должны предоставить определённый результат: код, отчёт, скрипт или демонстрацию.  
**Тема проекта может быть любой** (переводчик, генератор сказок, QA-система, анализатор резюме и т.д.) – главное, чтобы в ней применялись все изученные технологии и архитектурные паттерны.

---

## **Этап 1: Выбор темы и настройка окружения**

**Цель:** Определиться с предметной областью и подготовить рабочее место.

**Задачи:**
- Выбрать тему проекта и чётко сформулировать, какую задачу будет решать система.
- Сформировать команду (2–3 человека), распределить роли.
- Изучить требования к проекту (этот план).
- Создать удалённый репозиторий на GitHub или GitLab.
- Настроить Poetry: создать `pyproject.toml` со всеми необходимыми зависимостями (перечислены в общем задании).
- Добавить `.gitignore` и написать черновик README (описание проекта, цели, стек).

**Результат:**  
Репозиторий с Poetry, базовым `pyproject.toml`, выбранной темой и первичным README.

**Критерии готовности:**
- [ ] Тема выбрана и обоснована.
- [ ] Poetry устанавливает зависимости без ошибок.
- [ ] README содержит краткое описание проекта.

---

## **Этап 2: Проектирование архитектуры и каркас приложения**

**Цель:** Создать скелет проекта и базовые модули.

**Задачи:**
- Спроектировать структуру папок (можно взять за основу предложенную в задании, адаптировав под свою тему).
- Создать `backend/core/config.py` с использованием `pydantic-settings` – все настройки: БД, LLM, Qdrant, Redis, JWT.
- Добавить `backend/core/constants.py` – константы (например, поддерживаемые категории, языки, типы запросов).
- Создать `backend/core/exceptions.py` – пользовательские классы исключений.
- Реализовать `backend/api/main.py` – инициализация FastAPI, подключение CORS, добавление простого эндпоинта `/healthz`.
- Написать `run.py` для запуска сервера (uvicorn).
- Настроить логирование (`backend/utils/logging.py`) с ротацией и разными уровнями.

**Результат:**  
Приложение запускается, отвечает на `/healthz`, структура папок полностью соответствует задуманной архитектуре.

**Критерии готовности:**
- [ ] Структура папок создана и соответствует описанию в README.
- [ ] Конфигурация читается из переменных окружения.
- [ ] Логирование выводит сообщения в консоль и файл.

---

## **Этап 3: База данных**

**Цель:** Настроить подключение к БД и создать модели данных.

**Задачи:**
- Реализовать `backend/db/session.py` – асинхронное подключение через SQLAlchemy (engine, async_sessionmaker).
- Создать `backend/db/models.py` со следующими таблицами (названия можно адаптировать под тему):
  - `users` – пользователи (id, email, username, hashed_password, role, is_active, created_at).
  - `knowledge_base` – база знаний (id, source_text, target_text, category, created_at) – аналог Translation Memory.
  - `glossary` – справочник/словарь терминов (id, term, definition/translation, category, created_at).
  - `uncertain_examples` – неоднозначные примеры для HITL (id, input_text, model_output, critic_score, status, created_at).
  - `cache` – кэш результатов (id, input_key, output, hits, created_at).
- Настроить автоматическое создание таблиц при старте приложения.
- Подключить PostgreSQL в docker-compose (локально можно использовать SQLite).
- Протестировать подключение и создание таблиц.

**Результат:**  
База данных инициализируется при запуске, все необходимые таблицы созданы.

**Критерии готовности:**
- [ ] Приложение успешно подключается к БД.
- [ ] Все таблицы создаются автоматически.
- [ ] Модели описаны корректно, с индексами и ограничениями.

---

## **Этап 4: Аутентификация и авторизация**

**Цель:** Реализовать регистрацию, вход и систему ролей.

**Задачи:**
- Создать `backend/utils/security.py` – функции для хеширования паролей (bcrypt) и работы с JWT (создание, проверка).
- Реализовать `backend/api/deps.py` – зависимости для получения текущего пользователя и проверки ролей (`get_current_user`, `require_admin`, `require_editor`).
- Создать роуты в `backend/api/routes/auth.py`:
  - `POST /auth/register` – регистрация нового пользователя (роль по умолчанию – `user`).
  - `POST /auth/login` – вход, возврат JWT-токена.
  - `GET /auth/me` – информация о текущем пользователе.
- Добавить Pydantic-модели для запросов и ответов.
- Написать юнит-тесты для регистрации, входа и получения информации (с мок-сессией).

**Результат:**  
Работающая аутентификация. Доступ к защищённым эндпоинтам имеют только авторизованные пользователи, а к административным – только обладатели нужных ролей.

**Критерии готовности:**
- [ ] Пароли хранятся только в виде bcrypt-хеша.
- [ ] JWT-токен выпускается и проверяется.
- [ ] Тесты на аутентификацию проходят.

---

## **Этап 5: Базовый класс агента и Supervisor**

**Цель:** Заложить основу мультиагентной системы.

**Задачи:**
- Создать `backend/agents/base.py` – абстрактный класс `Agent` с методом `async def process(state)`, хранящий имя агента и последний результат.
- Реализовать `backend/agents/supervisor.py` – класс `Supervisor`, который:
  - хранит ссылки на агентов,
  - лениво их инициализирует,
  - организует цикл: пока требуется коррекция и не превышено число итераций, последовательно вызывает агентов,
  - ведёт `reasoning_log` и счётчик итераций.
- Написать юнит-тесты для Supervisor, используя mock-агентов (проверить условие выхода, максимальное число итераций).

**Результат:**  
Рабочий Supervisor, способный управлять любыми агентами, соответствующими интерфейсу.

**Критерии готовности:**
- [ ] Базовый класс агента реализован.
- [ ] Supervisor корректно выполняет цикл итераций.
- [ ] Тесты Supervisor проходят.

---

## **Этап 6: Агент Validator**

**Цель:** Реализовать агента формальной проверки.

**Задачи:**
- Создать `backend/agents/validator.py`, наследующийся от `Agent`.
- Реализовать набор правил (в зависимости от темы):
  - минимальная длина результата,
  - наличие обязательных элементов,
  - отсутствие дубликатов,
  - проверка по справочнику (если применимо) и т.п.
- При обнаружении ошибок/предупреждений агент может вызвать LLM для дополнительного анализа (опционально на этом этапе, можно использовать заглушку).
- Записывать в `state` списки `errors`, `warnings` и флаг `validation_passed`.
- Написать юнит-тесты: проверка срабатывания правил, корректность возвращаемых данных.

**Результат:**  
Агент Validator работает, умеет находить формальные недостатки и передавать их дальше.

**Критерии готовности:**
- [ ] Все запланированные правила реализованы.
- [ ] Агент возвращает корректные `errors` и `warnings`.
- [ ] Тесты Validator проходят.

---

## **Этап 7: Агент Critic (LLM-as-a-Judge)**

**Цель:** Реализовать агента оценки качества.

**Задачи:**
- Создать `backend/agents/critic.py`.
- Реализовать формирование промпта с few-shot примерами (хороший и плохой ответ для вашей задачи).
- Вызывать LLM (пока можно использовать заглушку, реальное подключение – на этапе 11).
- Парсить JSON-ответ с оценками по нескольким критериям (например, релевантность, полнота, стиль, общий балл).
- Реализовать fallback-логику: если LLM вернул ошибку или невалидный JSON, возвращать средние оценки (например, 5.0) и пояснение.
- Записывать в `state`: `critic_score`, `critic_reasoning`, `critic_suggestions`, `critic_passed`.
- Написать юнит-тесты: проверка промпта, парсинг JSON (включая markdown-блоки), обработка ошибок.

**Результат:**  
Агент Critic умеет оценивать качество результата и выдавать рекомендации.

**Критерии готовности:**
- [ ] Промпт содержит понятные инструкции и примеры.
- [ ] Оценки извлекаются корректно.
- [ ] Fallback-механизм работает при ошибках.

---

## **Этап 8: Агент Editor**

**Цель:** Реализовать агента исправления/улучшения.

**Задачи:**
- Создать `backend/agents/editor.py`.
- Агент принимает ошибки от Validator и рекомендации от Critic.
- Вызывает LLM с промптом на исправление/улучшение результата.
- Обновляет основной результат в `state` (например, `state["translation"]` или `state["answer"]`).
- Устанавливает флаг `editor_changed` (изменился ли результат).
- Если ошибок и рекомендаций нет, агент не вызывает LLM.
- Написать юнит-тесты: вызов LLM при наличии проблем, отсутствие вызова при пустых входных данных, обработка исключений.

**Результат:**  
Агент Editor умеет улучшать результат на основе замечаний.

**Критерии готовности:**
- [ ] Агент вызывается только при необходимости.
- [ ] Результат обновляется корректно.
- [ ] Тесты Editor проходят.

---

## **Этап 9: Сбор и импорт эталонных данных (глоссарий и ТМ)**

**Цель:** Подготовить датасет и загрузить его в систему.

**Задачи:**
- Найти или создать исходные данные. Для перевода это пары предложений, для QA – вопросы и ответы, для генерации сказок – сюжеты и тексты, для анализа резюме – резюме и вакансии и т.д.
- Написать скрипт генерации датасета (например, `scripts/generate_data.py`), который:
  - читает исходные файлы (JSON, CSV, txt),
  - сэмплирует нужное количество записей:
    - **для базы знаний (аналог TM) – не менее 1000 пар**,
    - **для справочника/глоссария – не менее 500 уникальных записей**,
  - для глоссария можно выделить частотные термины (слова или фразы) и их соответствия,
  - сохраняет результат в CSV или JSON.
- Написать скрипт импорта данных в БД (`scripts/import_data.py`), используя SQLAlchemy, с защитой от дубликатов (например, проверка на существование записи).
- Загрузить данные в локальную БД и проверить, что все записи появились.

**Результат:**  
В базе данных содержится не менее 1500 записей (1000 в базе знаний + 500 в справочнике). Скрипты генерации и импорта работают.

**Критерии готовности:**
- [ ] Датасет подготовлен и сохранён в нужном формате.
- [ ] Импорт в БД выполняется без дубликатов.
- [ ] Объёмы данных соответствуют требованиям.

---

## **Этап 10: RAG – индексы и гибридный поиск**

**Цель:** Реализовать поиск по базе знаний.

**Задачи:**
- Создать `backend/data/embeddings.py` – класс-синглтон для модели эмбеддингов (например, SentenceTransformer).
- Создать `backend/retrieval/bm25_index.py` – обёртку над `rank_bm25` для лексического поиска.
- Создать `backend/retrieval/vector_index.py` – класс для работы с Qdrant: создание коллекции, добавление точек, поиск с фильтрами.
- Создать `backend/retrieval/hybrid_retriever.py` – объединение результатов BM25 и векторного поиска через Reciprocal Rank Fusion (RRF).
- Написать юнит-тесты для каждого индекса и для гибридного поиска (BM25 – с реальными данными, векторный – с мок-клиентом Qdrant, RRF – на синтетических результатах).

**Результат:**  
Гибридный поиск находит релевантные записи по запросу, комбинируя лексическое и семантическое сходство.

**Критерии готовности:**
- [ ] Эмбеддинги вычисляются и кэшируются (модель загружается один раз).
- [ ] BM25 индекс строится и ищет.
- [ ] Векторный поиск работает с Qdrant.
- [ ] RRF возвращает объединённый список.

---

## **Этап 11: Интеграция LLM (Ollama и HuggingFace)**

**Цель:** Подключить реальные LLM и настроить выбор бэкенда.

**Задачи:**
- Реализовать `backend/utils/llm_client.py` – асинхронный клиент для Ollama (HTTP API, таймауты, повторные попытки, очистка ответа).
- Реализовать обёртку для HuggingFace Transformers (`backend/data/hf_model.py`), использующую `transformers.pipeline` и кэширование результатов.
- Создать менеджер выбора (`backend/data/llm_manager.py`): если задача/язык/модель поддерживается Ollama – использовать его, иначе – HuggingFace.
- Обновить агентов (Validator, Critic, Editor), чтобы они использовали реальный LLM-клиент вместо заглушек.
- Настроить обработку ошибок: таймауты, повторы, fallback.
- Написать тесты с мок-httpx для Ollama и мок-pipeline для HF.

**Результат:**  
Система умеет обращаться к LLM через оба бэкенда и автоматически выбирает подходящий.

**Критерии готовности:**
- [ ] Клиент Ollama работает (проверено на реальном сервере или моке).
- [ ] HuggingFace pipeline вызывается асинхронно.
- [ ] Агенты переведены на реальные LLM.

---

## **Этап 12: Основные API эндпоинты и интеграция всех компонентов**

**Цель:** Связать агентов, RAG и LLM через REST API.

**Задачи:**
- Реализовать основной роут (например, `backend/api/routes/generate.py`):
  - принимает входные данные,
  - проверяет кэш,
  - в зависимости от выбранного режима (baseline / full) либо просто вызывает LLM, либо запускает Supervisor с агентами,
  - возвращает результат вместе с метаданными: оценки, логи, использованные записи из базы знаний.
- Добавить дополнительные роуты:
  - пакетная обработка,
  - мульти-варианты (несколько целевых языков/категорий),
  - ансамбль моделей (запуск нескольких LLM и выбор лучшего по оценке Critic).
- Создать административные роуты для управления справочником и базой знаний (CRUD, импорт CSV).
- Написать интеграционные тесты API с `TestClient` и моками LLM/поиска.

**Результат:**  
Полноценное API, которое выполняет основную задачу, используя мультиагентную систему и RAG.

**Критерии готовности:**
- [ ] Основной эндпоинт возвращает корректный результат.
- [ ] Режимы baseline и full работают.
- [ ] Админ-эндпоинты позволяют управлять данными.

---

## **Этап 13: Human-in-the-Loop и Active Learning**

**Цель:** Реализовать механизм обратной связи и обработки неоднозначных результатов.

**Задачи:**
- Создать `backend/hitl/active_learning.py`:
  - метод добавления примера в очередь, когда качество низкое (например, `critic_score < 7` или `validation_passed == False`),
  - метод получения списка ожидающих примеров,
  - метод подтверждения исправления (сохраняет исправленный результат в базу знаний),
  - метод отклонения примера.
- Создать `backend/hitl/feedback.py` – сохранение исправлений от пользователя.
- Добавить соответствующие роуты `/hitl/...`.
- Написать юнит-тесты для всех методов.

**Результат:**  
Пользователь с ролью `editor` или `admin` может просматривать неоднозначные примеры, подтверждать или отклонять их. Подтверждённые исправления пополняют базу знаний.

**Критерии готовности:**
- [ ] Очередь неоднозначных примеров создаётся.
- [ ] Подтверждение обновляет базу знаний.
- [ ] Роуты защищены соответствующими ролями.

---

## **Этап 14: Celery задачи и кэширование**

**Цель:** Добавить фоновую обработку и оптимизацию.

**Задачи:**
- Настроить `backend/worker/celery_app.py` с брокером Redis.
- Реализовать задачи в `backend/worker/tasks.py`:
  - переиндексация базы знаний (обновление BM25 и Qdrant),
  - пакетная обработка запросов,
  - очистка устаревшего кэша,
  - массовое добавление записей.
- Подключить кэш (таблица в БД или Redis) для результатов: перед вызовом LLM проверять кэш, после успешного выполнения – сохранять.
- Увеличивать счётчик попаданий в кэше.
- Написать тест для одной из задач (например, с `task_always_eager`).

**Результат:**  
Тяжёлые операции выполняются в фоне, результаты кэшируются, система стала быстрее.

**Критерии готовности:**
- [ ] Celery worker запускается и выполняет задачи.
- [ ] Кэш используется, счётчик попаданий увеличивается.
- [ ] Периодическая задача очистки кэша настроена.

---

## **Этап 15: Frontend (Streamlit) и финальная сборка**

**Цель:** Создать пользовательский интерфейс и подготовить проект к сдаче.

**Задачи:**
- Настроить Streamlit-приложение (`frontend/app.py` или `streamlit_app/app.py`):
  - страница входа/регистрации,
  - основная страница с формой для выполнения задачи,
  - страница пакетной обработки (загрузка файла),
  - страница управления справочником и базой знаний (CRUD, импорт CSV),
  - страница HITL (просмотр и обработка неоднозначных примеров),
  - страница мониторинга (статусы сервисов, статистика кэша).
- Написать модуль `api_client.py` для взаимодействия с FastAPI.
- Создать Dockerfile и docker-compose.yml со всеми сервисами: `app`, `worker`, `beat`, `db`, `redis`, `qdrant`, `ollama`, `nginx`.
- Настроить Nginx (проксирование, сжатие, заголовки безопасности).
- Провести финальное тестирование: запустить все контейнеры, проверить healthcheck'и и работоспособность ключевых сценариев.
- Обновить README: инструкция по запуску одной командой, описание архитектуры, скриншоты.

**Результат:**  
Готовый проект, который можно запустить на любой машине с Docker одной командой. UI позволяет пользоваться всеми функциями системы.

**Критерии готовности:**
- [ ] Все страницы Streamlit работают.
- [ ] Docker Compose поднимает все сервисы без ошибок.
- [ ] README содержит понятную инструкцию.
- [ ] Демонстрация основных сценариев проходит успешно.

---

## 📌 **Общие замечания**

- Каждый этап завершается коммитом в Git и коротким отчётом (что сделано, с какими трудностями столкнулись).
- Рекомендуется использовать ветки для каждого этапа (например, `week-1`, `week-2` и т.д.).
- Тесты пишутся сразу вместе с кодом (TDD приветствуется).
- В конце курса – презентация проекта и ответы на вопросы.

**Удачи!** 🚀